In [1]:
# MPAGE Package Testing
# This notebook demonstrates basic functionality of the MPAGE package

# Load the package
devtools::load_all()


ℹ Loading MPAGE




Warning message:
“Objects listed as exports, but not present in namespace:
• add_custom_proteins
• filter_ppi_network
• load_ppi_network
• save_ppi_network
• save_rna_mod_proteins
• preprocess_intact_data
• load_intact_network
• download_intact_data”


In [2]:
networkds <- build_ppi_network()


Processing STRING database...



STRING Network Summary:


Processed 6857702 unique interactions

Unique proteins: 19622



Nodes: 16185 
Edges: 236000 
Average degree: 29.16281 


Processing BIOGRID database...



...

Nodes: 20169 
Edges: 959426 
Average degree: 95.13868 


Processing INTACT database...



INTACT Network Summary:


Processed 606332 unique interactions

Unique proteins: 18416



Nodes: 5597 
Edges: 20870 
Average degree: 7.457567 


In [3]:
# test <- preprocessing_string(
#     species_code = 9606,
#     score_threshold = 0,
#     output_file = "inst/extdata/stringdb_9606_v12.rds",
#     version = "12",
#     verbose = TRUE
# )


In [4]:
biogrid_file <- "data-raw/BIOGRID-ALL-4.4.248.tab3.txt"
biogrid_data <- data.table::fread(biogrid_file, sep = "\t", check.names = TRUE, quote = "")


In [5]:
verbose <- TRUE


In [6]:
if (verbose) {
  message(sprintf("Loaded %d interactions from BioGRID", nrow(biogrid_data)))
}


Loaded 2804534 interactions from BioGRID



In [7]:
biogrid_data %>% colnames()


[1] "X.BioGRID.Interaction.ID"           "Entrez.Gene.Interactor.A"          
 [3] "Entrez.Gene.Interactor.B"           "BioGRID.ID.Interactor.A"           
 [5] "BioGRID.ID.Interactor.B"            "Systematic.Name.Interactor.A"      
 [7] "Systematic.Name.Interactor.B"       "Official.Symbol.Interactor.A"      
 [9] "Official.Symbol.Interactor.B"       "Synonyms.Interactor.A"             
[11] "Synonyms.Interactor.B"              "Experimental.System"               
[13] "Experimental.System.Type"           "Author"                            
[15] "Publication.Source"                 "Organism.ID.Interactor.A"          
[17] "Organism.ID.Interactor.B"           "Throughput"                        
[19] "Score"                              "Modification"                      
[21] "Qualifications"                     "Tags"                              
[23] "Source.Database"                    "SWISS.PROT.Accessions.Interactor.A"
[25] "TREMBL.Accessions.Interactor.A"     "REFSEQ.Accessions.Interactor.A"    
[27] "SWISS.PROT.Accessions.Interactor.B" "TREMBL.Accessions.Interactor.B"    
[29] "REFSEQ.Accessions.Interactor.B"     "Ontology.Term.IDs"                 
[31] "Ontology.Term.Names"                "Ontology.Term.Categories"          
[33] "Ontology.Term.Qualifier.IDs"        "Ontology.Term.Qualifier.Names"     
[35] "Ontology.Term.Types"                "Organism.Name.Interactor.A"        
[37] "Organism.Name.Interactor.B"

In [8]:
processed_data <- biogrid_data %>%
  dplyr::select(
    gene1 = Official.Symbol.Interactor.A,
    gene2 = Official.Symbol.Interactor.B,
    species_codeA = Organism.ID.Interactor.A,
    species_codeB = Organism.ID.Interactor.B,
    Experimental.System.Type
  ) %>%
  dplyr::filter(species_codeA == species_codeB, species_codeA == 9606)



if (verbose) {
  message(sprintf("Processed %d unique interactions", nrow(processed_data)))
  message(sprintf("Unique proteins: %d", length(unique(c(processed_data$gene1, processed_data$gene2)))))
}


Processed 1219381 unique interactions

Unique proteins: 20334



In [9]:
biogrid_data %>% colnames()


[1] "X.BioGRID.Interaction.ID"           "Entrez.Gene.Interactor.A"          
 [3] "Entrez.Gene.Interactor.B"           "BioGRID.ID.Interactor.A"           
 [5] "BioGRID.ID.Interactor.B"            "Systematic.Name.Interactor.A"      
 [7] "Systematic.Name.Interactor.B"       "Official.Symbol.Interactor.A"      
 [9] "Official.Symbol.Interactor.B"       "Synonyms.Interactor.A"             
[11] "Synonyms.Interactor.B"              "Experimental.System"               
[13] "Experimental.System.Type"           "Author"                            
[15] "Publication.Source"                 "Organism.ID.Interactor.A"          
[17] "Organism.ID.Interactor.B"           "Throughput"                        
[19] "Score"                              "Modification"                      
[21] "Qualifications"                     "Tags"                              
[23] "Source.Database"                    "SWISS.PROT.Accessions.Interactor.A"
[25] "TREMBL.Accessions.Interactor.A"     "REFSEQ.Accessions.Interactor.A"    
[27] "SWISS.PROT.Accessions.Interactor.B" "TREMBL.Accessions.Interactor.B"    
[29] "REFSEQ.Accessions.Interactor.B"     "Ontology.Term.IDs"                 
[31] "Ontology.Term.Names"                "Ontology.Term.Categories"          
[33] "Ontology.Term.Qualifier.IDs"        "Ontology.Term.Qualifier.Names"     
[35] "Ontology.Term.Types"                "Organism.Name.Interactor.A"        
[37] "Organism.Name.Interactor.B"

In [10]:
species_code <- 9606
processed_data <- biogrid_data %>%
  dplyr::select(
    gene1 = Official.Symbol.Interactor.A,
    gene2 = Official.Symbol.Interactor.B,
    species_codeA = Organism.ID.Interactor.A,
    species_codeB = Organism.ID.Interactor.B,
    Experimental.System.Type
  ) %>%
  dplyr::mutate(species_code = species_code, est = factor(Experimental.System.Type, levels = c("physical", "genetic"))) %>%
  dplyr::filter(species_codeA == species_codeB, species_codeA == species_code) %>%
  arrange(est) %>%
  distinct(gene1, gene2, .keep_all = TRUE) %>%
  dplyr::select(gene1, gene2, species_code, est)


In [11]:
saveRDS(processed_data, "inst/extdata/biogrid_9606_v4.4.248.rds")


In [12]:
readRDS("inst/extdata/intact_9606_v20250328.rds")


gene1,gene2,species_code,score
<chr>,<chr>,<chr>,<dbl>
MDM2,TP53,9606,1.00
TP53,MDM2,9606,1.00
CDK4,CCND1,9606,0.99
CCND1,CDK4,9606,0.99
CCNT1,CDK9,9606,0.98
EIF4EBP1,EIF4E,9606,0.98
BCL2L1,BAD,9606,0.98
CASP8,FADD,9606,0.98
HRAS,RAF1,9606,0.98


In [13]:
processed_data <- biogrid_data %>%
  dplyr::select(
    gene1 = Official.Symbol.Interactor.A,
    gene2 = Official.Symbol.Interactor.B,
    species_codeA = Organism.ID.Interactor.A,
    species_codeB = Organism.ID.Interactor.B,
    Experimental.System.Type
  ) %>%
  dplyr::filter(species_codeA == species_codeB, species_codeA == 9606)



if (verbose) {
  message(sprintf("Processed %d unique interactions", nrow(processed_data)))
  message(sprintf("Unique proteins: %d", length(unique(c(processed_data$gene1, processed_data$gene2)))))
}


Processed 1219381 unique interactions

Unique proteins: 20334



In [14]:
biogrid_data %>% colnames()


[1] "X.BioGRID.Interaction.ID"           "Entrez.Gene.Interactor.A"          
 [3] "Entrez.Gene.Interactor.B"           "BioGRID.ID.Interactor.A"           
 [5] "BioGRID.ID.Interactor.B"            "Systematic.Name.Interactor.A"      
 [7] "Systematic.Name.Interactor.B"       "Official.Symbol.Interactor.A"      
 [9] "Official.Symbol.Interactor.B"       "Synonyms.Interactor.A"             
[11] "Synonyms.Interactor.B"              "Experimental.System"               
[13] "Experimental.System.Type"           "Author"                            
[15] "Publication.Source"                 "Organism.ID.Interactor.A"          
[17] "Organism.ID.Interactor.B"           "Throughput"                        
[19] "Score"                              "Modification"                      
[21] "Qualifications"                     "Tags"                              
[23] "Source.Database"                    "SWISS.PROT.Accessions.Interactor.A"
[25] "TREMBL.Accessions.Interactor.A"     "REFSEQ.Accessions.Interactor.A"    
[27] "SWISS.PROT.Accessions.Interactor.B" "TREMBL.Accessions.Interactor.B"    
[29] "REFSEQ.Accessions.Interactor.B"     "Ontology.Term.IDs"                 
[31] "Ontology.Term.Names"                "Ontology.Term.Categories"          
[33] "Ontology.Term.Qualifier.IDs"        "Ontology.Term.Qualifier.Names"     
[35] "Ontology.Term.Types"                "Organism.Name.Interactor.A"        
[37] "Organism.Name.Interactor.B"

In [15]:
# Process the data to required format
# BioGRID format processing
processed_data <- biogrid_data %>%
  dplyr::select(
    gene1 = Official.Symbol.Interactor.A,
    gene2 = Official.Symbol.Interactor.B,
    species_codeA = Organism.ID.Interactor.A,
    species_codeB = Organism.ID.Interactor.B,
    Score = Score
  ) %>%
  dplyr::mutate(
    # Convert gene IDs to character and clean
    geneA = as.character(.data$geneA),
    geneB = as.character(.data$geneB),

    # Ensure species codes are character
    species_codeA = as.character(.data$species_codeA),
    species_codeB = as.character(.data$species_codeB),

    # BioGRID doesn't have confidence scores, use 1.0 as default
    score = 1.0,

    # Filter for specified species
    species_code = .data$species_codeA
  ) %>%
  # Filter for specified species
  dplyr::filter(
    .data$species_codeA == species_code,
    .data$species_codeB == species_code,
    .data$species_codeA == .data$species_codeB
  ) %>%
  # Clean and finalize
  dplyr::mutate(
    geneA = toupper(.data$geneA),
    geneB = toupper(.data$geneB)
  ) %>%
  dplyr::select(geneA, geneB, score, species_code) %>%
  dplyr::distinct() %>%
  dplyr::filter(
    !is.na(.data$geneA),
    !is.na(.data$geneB),
    .data$geneA != .data$geneB,
    .data$geneA != "",
    .data$geneB != "",
    .data$geneA != "-",
    .data$geneB != "-"
  )

# Clean up temporary files
unlink(temp_zip)
unlink(extract_dir, recursive = TRUE)

if (verbose) {
  message(sprintf("Processed %d unique interactions", nrow(processed_data)))
  message(sprintf("Unique proteins: %d", length(unique(c(processed_data$geneA, processed_data$geneB)))))
}

# Save if output file specified
if (!is.null(output_file)) {
  saveRDS(processed_data, file = output_file)
  if (verbose) {
    message(sprintf("Data saved to: %s", output_file))
  }
}

return(processed_data)


ERROR: [1m[33mError[39m in `dplyr::mutate()`:[22m
[1m[22m[36mℹ[39m In argument: `geneA = as.character(.data$geneA)`.
[1mCaused by error in `.data$geneA`:[22m
[1m[22m[33m![39m Column `geneA` not found in `.data`.


In [ ]:
a <- list(
  string_version = "12"
)


In [ ]:
a[["string_version"]]


[1] "12"

In [ ]:

    
    if (verbose) {
      message(sprintf("Loaded %d interactions from BioGRID", nrow(biogrid_data)))
    }
    
    # Process the data to required format
    # BioGRID format processing
    processed_data <- biogrid_data %>%
      dplyr::select(
        # BioGRID column names
        geneA = .data$`Entrez Gene Interactor A`,
        geneB = .data$`Entrez Gene Interactor B`,
        species_codeA = .data$`Organism ID Interactor A`,
        species_codeB = .data$`Organism ID Interactor B`,
        experimental_system = .data$`Experimental System`,
        pubmed_id = .data$`Pubmed ID`
      ) %>%
      dplyr::mutate(
        # Convert gene IDs to character and clean
        geneA = as.character(.data$geneA),
        geneB = as.character(.data$geneB),
        
        # Ensure species codes are character
        species_codeA = as.character(.data$species_codeA),
        species_codeB = as.character(.data$species_codeB),
        
        # BioGRID doesn't have confidence scores, use 1.0 as default
        score = 1.0,
        
        # Filter for specified species
        species_code = .data$species_codeA
      ) %>%
      # Filter for specified species
      dplyr::filter(
        .data$species_codeA == species_code,
        .data$species_codeB == species_code,
        .data$species_codeA == .data$species_codeB
      ) %>%
      # Clean and finalize
      dplyr::mutate(
        geneA = toupper(.data$geneA),
        geneB = toupper(.data$geneB)
      ) %>%
      dplyr::select(geneA, geneB, score, species_code) %>%
      dplyr::distinct() %>%
      dplyr::filter(
        !is.na(.data$geneA),
        !is.na(.data$geneB),
        .data$geneA != .data$geneB,
        .data$geneA != "",
        .data$geneB != "",
        .data$geneA != "-",
        .data$geneB != "-"
      )
    
    # Clean up temporary files
    unlink(temp_zip)
    unlink(extract_dir, recursive = TRUE)
    
    if (verbose) {
      message(sprintf("Processed %d unique interactions", nrow(processed_data)))
      message(sprintf("Unique proteins: %d", length(unique(c(processed_data$geneA, processed_data$geneB)))))
    }
    
    # Save if output file specified
    if (!is.null(output_file)) {
      saveRDS(processed_data, file = output_file)
      if (verbose) {
        message(sprintf("Data saved to: %s", output_file))
      }
    }
    
    return(processed_data)
    
  }, error = function(e) {
    stop("Error processing BioGRID data: ", e$message)
  })
}


In [ ]:
library(dplyr)



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [ ]:
species_code <- 9606


In [ ]:
# Process the data to required format
# IntAct PSI-MITAB format processing
processed_data <- intact_data %>%
    dplyr::select(
        Alt..ID.s..interactor.A,
        Alt..ID.s..interactor.B,
        Alias.es..interactor.A,
        Alias.es..interactor.B,
        Taxid.interactor.A,
        Taxid.interactor.B,
        Confidence.value.s.
    ) %>%
    # Extract species information
    dplyr::mutate(
        taxidA = stringr::str_extract(Taxid.interactor.A, "taxid:([0-9]+)") %>%
            stringr::str_remove("taxid:"),
        taxidB = stringr::str_extract(Taxid.interactor.B, "taxid:([0-9]+)") %>%
            stringr::str_remove("taxid:")
    ) %>%
    # Filter by species
    dplyr::filter(
        !is.na(taxidA), !is.na(taxidB),
        taxidA == species_code,
        taxidB == species_code,
        taxidA == taxidB
    ) %>%
    # Extract gene names using multiple patterns
    dplyr::mutate(
        # Extract from gene name annotations
        geneA_name = stringr::str_extract(
            Alias.es..interactor.A,
            "uniprotkb:[A-Za-z0-9-]+\\(gene name\\)"
        ) %>%
            stringr::str_remove("uniprotkb:") %>%
            stringr::str_remove("\\(gene name\\)") %>%
            toupper(),
        geneB_name = stringr::str_extract(
            Alias.es..interactor.B,
            "uniprotkb:[A-Za-z0-9-]+\\(gene name\\)"
        ) %>%
            stringr::str_remove("uniprotkb:") %>%
            stringr::str_remove("\\(gene name\\)") %>%
            toupper(),

        # Extract from display names as fallback
        geneA_display = stringr::str_extract(
            Alias.es..interactor.A,
            "display_short\\):([^|]+)"
        ) %>%
            stringr::str_remove("display_short\\):"),
        geneB_display = stringr::str_extract(
            Alias.es..interactor.B,
            "display_short\\):([^|]+)"
        ) %>%
            stringr::str_remove("display_short\\):"),

        # Use gene name if available, otherwise display name
        gene1 = dplyr::coalesce(geneA_name, geneA_display),
        gene2 = dplyr::coalesce(geneB_name, geneB_display),

        # Extract interaction score
        score = stringr::str_extract(Confidence.value.s., "intact-miscore:([0-9\\.]+)") %>%
            stringr::str_remove("intact-miscore:") %>%
            as.numeric(),
        species_code = taxidA
    ) %>%
    # Clean and filter
    dplyr::select(gene1, gene2, score, species_code) %>%
    dplyr::filter(
        !is.na(gene1), !is.na(gene2), gene1 != "", gene2 != "", gene1 != gene2,
        !is.na(score),
    ) %>%
    dplyr::distinct() %>%
    dplyr::arrange(dplyr::desc(score)) %>%
    dplyr::select(gene1, gene2, species_code, score)


In [ ]:
saveRDS(processed_data, "inst/extdata/intact_9606_v20250328.rds")


In [ ]:
if (verbose) {
    message(sprintf("Loaded %d interactions from IntAct", nrow(intact_data)))
}

# Process the data to required format
# IntAct PSI-MITAB format processing
processed_data <- intact_data %>%
    dplyr::select(
        Alt..ID.s..interactor.A,
        Alt..ID.s..interactor.B,
        Alias.es..interactor.A,
        Alias.es..interactor.B,
        Taxid.interactor.A,
        Taxid.interactor.B,
        Confidence.value.s.
    ) %>%
    # Extract species information
    dplyr::mutate(
        taxidA = stringr::str_extract(Taxid.interactor.A, "taxid:([0-9]+)") %>%
            stringr::str_remove("taxid:"),
        taxidB = stringr::str_extract(Taxid.interactor.B, "taxid:([0-9]+)") %>%
            stringr::str_remove("taxid:")
    ) %>%
    # Filter by species
    dplyr::filter(
        !is.na(taxidA), !is.na(taxidB),
        taxidA %in% species_filter,
        taxidB %in% species_filter,
        taxidA == taxidB
    ) %>%
    # Extract gene names using multiple patterns
    dplyr::mutate(
        # Extract from gene name annotations
        geneA_name = stringr::str_extract(
            Alias.es..interactor.A,
            "uniprotkb:[A-Za-z0-9-]+\\(gene name\\)"
        ) %>%
            stringr::str_remove("uniprotkb:") %>%
            stringr::str_remove("\\(gene name\\)") %>%
            toupper(),
        geneB_name = stringr::str_extract(
            Alias.es..interactor.B,
            "uniprotkb:[A-Za-z0-9-]+\\(gene name\\)"
        ) %>%
            stringr::str_remove("uniprotkb:") %>%
            stringr::str_remove("\\(gene name\\)") %>%
            toupper(),

        # Extract from display names as fallback
        geneA_display = stringr::str_extract(
            Alias.es..interactor.A,
            "display_short\\):([^|]+)"
        ) %>%
            stringr::str_remove("display_short\\):"),
        geneB_display = stringr::str_extract(
            Alias.es..interactor.B,
            "display_short\\):([^|]+)"
        ) %>%
            stringr::str_remove("display_short\\):"),

        # Use gene name if available, otherwise display name
        geneA = dplyr::coalesce(geneA_name, geneA_display),
        geneB = dplyr::coalesce(geneB_name, geneB_display),

        # Extract interaction score
        score = stringr::str_extract(Confidence.value.s., "intact-miscore:([0-9\\.]+)") %>%
            stringr::str_remove("intact-miscore:") %>%
            as.numeric(),
        taxid = taxidA
    ) %>%
    # Clean and filter
    dplyr::select(geneA, geneB, score, taxid) %>%
    dplyr::filter(
        !is.na(geneA), !is.na(geneB), geneA != "", geneB != "",
        !is.na(score), score >= min_score
    ) %>%
    dplyr::distinct() %>%
    dplyr::arrange(dplyr::desc(score))
dplyr::mutate(
    # Extract UniProt IDs from complex identifiers
    geneA = stringr::str_extract(.data$geneA, "uniprotkb:([A-Za-z0-9-]+)") %>%
        stringr::str_remove("uniprotkb:"),
    geneB = stringr::str_extract(.data$geneB, "uniprotkb:([A-Za-z0-9-]+)") %>%
        stringr::str_remove("uniprotkb:"),

    # Extract species codes
    species_codeA = stringr::str_extract(.data$taxidA, "taxid:([0-9]+)") %>%
        stringr::str_remove("taxid:"),
    species_codeB = stringr::str_extract(.data$taxidB, "taxid:([0-9]+)") %>%
        stringr::str_remove("taxid:"),

    # Extract IntAct confidence score
    score = stringr::str_extract(.data$score_raw, "intact-miscore:([0-9\\.]+)") %>%
        stringr::str_remove("intact-miscore:") %>%
        as.numeric()
) %>%
    # Filter for specified species
    dplyr::filter(
        .data$species_codeA == species_code,
        .data$species_codeB == species_code,
        .data$species_codeA == .data$species_codeB
    ) %>%
    # Filter by score threshold
    dplyr::filter(
        !is.na(.data$score),
        .data$score >= score_threshold
    ) %>%
    # Clean and finalize
    dplyr::mutate(
        species_code = .data$species_codeA,
        geneA = toupper(.data$geneA),
        geneB = toupper(.data$geneB)
    ) %>%
    dplyr::select(geneA, geneB, score, species_code) %>%
    dplyr::distinct() %>%
    dplyr::filter(
        !is.na(.data$geneA),
        !is.na(.data$geneB),
        .data$geneA != .data$geneB,
        .data$geneA != "",
        .data$geneB != ""
    )


In [ ]:
test <- preprocessing_intact(
    species_code = "9606",
    score_threshold = 0.4,
    output_file = NULL,
    version = "2025-03-28",
    verbose = TRUE
)




To the file: /tmp/RtmpPOUGqD/file6fc22d54fd70.txt



Warning message in utils::download.file(intact_url, temp_file, quiet = !verbose, :
“downloaded length 21767737 != reported length 7890868954”
Warning message in utils::download.file(intact_url, temp_file, quiet = !verbose, :
“URL 'https://ftp.ebi.ac.uk/pub/databases/intact/2025-03-28/psimitab/intact.txt': Timeout of 60 seconds was reached”


ERROR: Error in utils::download.file(intact_url, temp_file, quiet = !verbose, : download from 'https://ftp.ebi.ac.uk/pub/databases/intact/2025-03-28/psimitab/intact.txt' failed


In [ ]:
download_ppi_data("STRING",
    output_dir = "data-raw/",
    processing_params = list(
        species_filter = c("9606"),
        min_score = 0.0
    ),
    overwrite = FALSE,
    verbose = TRUE
)


Warning message in download.file(config$url, paste0(temp_file, ".gz"), quiet = !verbose):
“downloaded length 0 != reported length 153”
Warning message in download.file(config$url, paste0(temp_file, ".gz"), quiet = !verbose):
“cannot open URL 'https://stringdb-static.org/download/protein.links.v12/9606.protein.links.v12.txt.gz': HTTP status was '404 Not Found'”
Warning message in value[[3L]](cond):
“Download failed for STRING: cannot open URL 'https://stringdb-static.org/download/protein.links.v12/9606.protein.links.v12.txt.gz'”
Warning message in value[[3L]](cond):
“Falling back to local file processing”


ERROR: Error in process_local_ppi_data(database_type, output_dir, processing_params, : Local STRING data file not found at: data-raw/string_interactions.txt


In [ ]:
df <- data.table::fread("data-raw/core.psimitab", sep = "\t", check.names = TRUE, quote = "", header = FALSE)


In [ ]:
df <- data.table::fread("data-raw/BIOGRID-ALL-4.4.248.tab3.txt", sep = "\t", check.names = TRUE, quote = "", header = TRUE)
df %>%
    dplyr::select(
        Official.Symbol.Interactor.A, Official.Symbol.Interactor.B,
        Organism.ID.Interactor.A,
        Organism.ID.Interactor.B, Organism.Name.Interactor.A, Organism.Name.Interactor.B, Throughput, Score, Modification,
        Experimental.System,
        Experimental.System.Type,
        Author,
        Publication.Source,
        Organism.ID.Interactor.A,
        Organism.ID.Interactor.B,
        Throughput,
        Score, Qualifications, Tags
    ) %>%
    dplyr::filter(Organism.ID.Interactor.A == Organism.ID.Interactor.B, Organism.ID.Interactor.A %in% c(9606, 10090), Organism.ID.Interactor.B %in% c(9606, 10090)) %>%
    arrange(Score) -> sadf
saveRDS(sadf, "inst/extdata/biogrid_processed.rds")


In [ ]:
df %>%
    pull(Experimental.System) %>%
    unique()


[1] "Two-hybrid"                    "Affinity Capture-MS"          
 [3] "Affinity Capture-Western"      "Dosage Rescue"                
 [5] "Reconstituted Complex"         "Synthetic Growth Defect"      
 [7] "Synthetic Lethality"           "Synthetic Rescue"             
 [9] "Affinity Capture-Luminescence" "Biochemical Activity"         
[11] "Co-crystal Structure"          "Far Western"                  
[13] "FRET"                          "Protein-peptide"              
[15] "Co-localization"               "Affinity Capture-RNA"         
[17] "Protein-RNA"                   "PCA"                          
[19] "Co-purification"               "Co-fractionation"             
[21] "Dosage Lethality"              "Phenotypic Enhancement"       
[23] "Phenotypic Suppression"        "Dosage Growth Defect"         
[25] "Negative Genetic"              "Positive Genetic"             
[27] "Synthetic Haploinsufficiency"  "Proximity Label-MS"           
[29] "Cross-Linking-MS (XL-MS)"      "Surface Display"

In [ ]:
df %>%
    dplyr::select(Throughput, Experimental.System, Experimental.System.Type, Score, Source.Database) %>%
    distinct()


Throughput,Experimental.System,Experimental.System.Type,Score,Source.Database
<chr>,<chr>,<chr>,<chr>,<chr>
Low Throughput,Two-hybrid,physical,-,BIOGRID
High Throughput,Two-hybrid,physical,-,BIOGRID
High Throughput,Affinity Capture-MS,physical,-,BIOGRID
Low Throughput,Affinity Capture-MS,physical,-,BIOGRID
Low Throughput,Affinity Capture-Western,physical,-,BIOGRID
Low Throughput,Dosage Rescue,genetic,-,BIOGRID
Low Throughput,Reconstituted Complex,physical,-,BIOGRID
High Throughput|Low Throughput,Synthetic Growth Defect,genetic,-,BIOGRID
Low Throughput,Synthetic Lethality,genetic,-,BIOGRID


In [ ]:
df %>% dplyr::select(
    Official.Symbol.Interactor.A, Official.Symbol.Interactor.B,
    Organism.ID.Interactor.A,
    Organism.ID.Interactor.B, Organism.Name.Interactor.A, Organism.Name.Interactor.B, Throughput, Score, Modification,
    Experimental.System,
    Experimental.System.Type,
    Author,
    Publication.Source,
    Organism.ID.Interactor.A,
    Organism.ID.Interactor.B,
    Throughput,
    Score, Qualifications, Tags
)


Official.Symbol.Interactor.A,Official.Symbol.Interactor.B,Organism.ID.Interactor.A,Organism.ID.Interactor.B,Organism.Name.Interactor.A,Organism.Name.Interactor.B,Throughput,Score,Modification,Experimental.System,Experimental.System.Type,Author,Publication.Source,Qualifications,Tags
<chr>,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
MAP2K4,FLNC,9606,9606,Homo sapiens,Homo sapiens,Low Throughput,-,-,Two-hybrid,physical,Marti A (1997),PUBMED:9006895,-,-
MYPN,ACTN2,9606,9606,Homo sapiens,Homo sapiens,Low Throughput,-,-,Two-hybrid,physical,Bang ML (2001),PUBMED:11309420,-,-
ACVR1,FNTA,9606,9606,Homo sapiens,Homo sapiens,Low Throughput,-,-,Two-hybrid,physical,Wang T (1996),PUBMED:8599089,-,-
GATA2,PML,9606,9606,Homo sapiens,Homo sapiens,Low Throughput,-,-,Two-hybrid,physical,Tsuzuki S (2000),PUBMED:10938104,-,-
RPA2,STAT3,9606,9606,Homo sapiens,Homo sapiens,Low Throughput,-,-,Two-hybrid,physical,Kim J (2000),PUBMED:10875894,-,-
ARF1,GGA3,9606,9606,Homo sapiens,Homo sapiens,Low Throughput,-,-,Two-hybrid,physical,Dell'Angelica EC (2000),PUBMED:10747089,-,-
ARF3,ARFIP2,9606,9606,Homo sapiens,Homo sapiens,Low Throughput,-,-,Two-hybrid,physical,Kanoh H (1997),PUBMED:9038142,-,-
ARF3,ARFIP1,9606,9606,Homo sapiens,Homo sapiens,Low Throughput,-,-,Two-hybrid,physical,Kanoh H (1997),PUBMED:9038142,-,-
XRN1,ALDOA,9606,9606,Homo sapiens,Homo sapiens,High Throughput,-,-,Two-hybrid,physical,Lehner B (2004),PUBMED:15231747,-,-


In [ ]:
# Build PPI network using STRING database
net <- build_ppi_network(
    proteins = NULL, # Use full network
    data_sources = c("STRING", "IntAct"),
    min_confidence = 0.7,
    species = "Homo sapiens",
    string_version = "12"
)



# Summary statistics
cat("Network Summary:\n")
cat("Nodes:", length(igraph::V(net)), "\n")
cat("Edges:", length(igraph::E(net)), "\n")
cat("Average degree:", mean(degree(net)), "\n")


ℹ Loading MPAGE




Warning message:
“Objects listed as exports, but not present in namespace:
• add_custom_proteins
• save_rna_mod_proteins”



STRING DB Network Summary:
Nodes: 16201 
Edges: 473860 
Average degree: 58.49762 


Loading IntAct network from: /workspaces/MPAGE/inst/extdata/intact_processed.rds



IntAct DB Network Summary:
Nodes: 5784 
Edges: 21915 
Average degree: 7.577801 
Network Summary:
Nodes: 16544 
Edges: 248963 
Average degree: 30.09707 



Loading BioGRID data from: /tmp/RtmpNNgF4Q/BIOGRID-ALL-4.4.248.tab3.txt

Warning message in FUN(file, exdir = tmpdir):
“error -1 in extracting from zip file”


ERROR: Error in value[[3L]](cond): Error reading BioGRID file: File is empty: /tmp/RtmpNNgF4Q/BIOGRID-ALL-4.4.248.tab3.txt


In [ ]:
get_string_ppi <- function(species_code, string_version, output_file) {
    tryCatch(
        {
            string_db <- STRINGdb::STRINGdb$new(
                species = species_code,
                version = string_version,
                score_threshold = 0
            )

            # Get full network instead of protein-limited network
            message("Downloading full STRING network for species ", species_code)
            stringdb_proteins <- string_db$get_proteins()
            # Get all interactions for the species above threshold
            all_interactions <- string_db$get_interactions(
                stringdb_proteins$protein_external_id
            )

            if (nrow(all_interactions) == 0) {
                return(NULL)
            }

            # Map interactions to gene symbols
            all_interactions$gene1 <- stringdb_proteins$preferred_name[match(all_interactions$from, stringdb_proteins$protein_external_id)]
            all_interactions$gene2 <- stringdb_proteins$preferred_name[match(all_interactions$to, stringdb_proteins$protein_external_id)]

            # Filter valid interactions
            all_interactions <- all_interactions[!is.na(all_interactions$gene1) & !is.na(all_interactions$gene2), ]

            saveRDS(all_interactions[, c("gene1", "gene2", "combined_score")], output_file)
            return(all_interactions[, c("gene1", "gene2", "combined_score")])
        },
        error = function(e) {
            warning("Error accessing STRING database: ", e$message)
            return(NULL)
        }
    )
}


In [ ]:
interactions <- get_string_ppi(9606, "12", "test.rds")


In [ ]:
species_code <- 9606
string_version <- "12"
sprintf("stringdb_%s_%s.rds", as.character(species_code), string_version)


[1] "stringdb_9606_12.rds"

In [ ]:
interactions[, c("gene1", "gene2", "combined_score")]


,gene1,gene2,combined_score
,<chr>,<chr>,<int>
1,ARF5,RALGPS2,173
2,ARF5,FHDC1,154
3,ARF5,ATP6V1E1,151
4,ARF5,CYTH2,471
5,ARF5,PSD3,201
6,ARF5,TTC9C,180
7,ARF5,SLC2A4,181
8,ARF5,GGA1,594
9,ARF5,RCC1L,154
